In [1]:
# Session Spark et schémas écrits à la main
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, DateType

spark = (
    SparkSession.builder
    .appName("TP3-Nettoyage")
    .master("local[*]")
    .getOrCreate()
)

def montant(nom):
    return StructField(nom, DecimalType(7, 2))

schema_ventes = StructType([
    StructField("ss_sold_date_sk", IntegerType()),
    StructField("ss_sold_time_sk", IntegerType()),
    StructField("ss_item_sk", IntegerType()),
    StructField("ss_customer_sk", IntegerType()),
    StructField("ss_cdemo_sk", IntegerType()),
    StructField("ss_hdemo_sk", IntegerType()),
    StructField("ss_addr_sk", IntegerType()),
    StructField("ss_store_sk", IntegerType()),
    StructField("ss_promo_sk", IntegerType()),
    StructField("ss_ticket_number", IntegerType()),
    StructField("ss_quantity", IntegerType()),
    montant("ss_wholesale_cost"), montant("ss_list_price"), montant("ss_sales_price"),
    montant("ss_ext_discount_amt"), montant("ss_ext_sales_price"), montant("ss_ext_wholesale_cost"),
    montant("ss_ext_list_price"), montant("ss_ext_tax"), montant("ss_coupon_amt"),
    montant("ss_net_paid"), montant("ss_net_paid_inc_tax"), montant("ss_net_profit"),
    StructField("_corrupt_record", StringType()),   # reçoit les lignes qui ne se convertissent pas
])

schema_produits = StructType([
    StructField("i_item_sk", IntegerType()),
    StructField("i_item_id", StringType()),
    StructField("i_rec_start_date", DateType()),
    StructField("i_rec_end_date", DateType()),
    StructField("i_item_desc", StringType()),
    montant("i_current_price"), montant("i_wholesale_cost"),
    StructField("i_brand_id", IntegerType()),
    StructField("i_brand", StringType()),
    StructField("i_class_id", IntegerType()),
    StructField("i_class", StringType()),
    StructField("i_category_id", IntegerType()),
    StructField("i_category", StringType()),
    StructField("i_manufact_id", IntegerType()),
    StructField("i_manufact", StringType()),
    StructField("i_size", StringType()),
    StructField("i_formulation", StringType()),
    StructField("i_color", StringType()),
    StructField("i_units", StringType()),
    StructField("i_container", StringType()),
    StructField("i_manager_id", IntegerType()),
    StructField("i_product_name", StringType()),
    StructField("_corrupt_record", StringType()),
])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:36:09 WARN Utils: Your hostname, DESKTOP-DCTNJ6T, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 15:36:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 15:36:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Lecture avec les schémas
def lire(chemin, schema):
    df = (
        spark.read
        .option("header", False)
        .option("sep", "|")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema)
        .csv(chemin)
    )
    return df.cache()

ventes_brut = lire("hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/store_sales/store_sales.dat", schema_ventes)
produits_brut = lire("hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/item/item.dat", schema_produits)

print("ventes lues :", ventes_brut.count(), "| produits lus :", produits_brut.count())
print("lignes mal converties - ventes :", ventes_brut.filter(F.col("_corrupt_record").isNotNull()).count(),
      "| produits :", produits_brut.filter(F.col("_corrupt_record").isNotNull()).count())
print("caractère \\r (fin de ligne Windows) resté dans la dernière colonne :",
      produits_brut.filter(F.col("i_product_name").contains("\r")).count())

ventes lues : 667579 | produits lus : 17820


lignes mal converties - ventes : 0 | produits : 0


caractère \r (fin de ligne Windows) resté dans la dernière colonne : 0


In [3]:
# Règles sur les produits
nb_meme_id = F.count("*").over(Window.partitionBy("i_item_sk"))

produits = (
    produits_brut
    .withColumn("nb_meme_id", nb_meme_id)
    .withColumn("motif_rejet",
        F.when(F.col("_corrupt_record").isNotNull(), "type incorrect")
         .when(F.col("i_item_sk").isNull(), "identifiant manquant")
         .when(F.col("nb_meme_id") > 1, "identifiant en double")
         .when(F.col("i_category").isNull(), "catégorie manquante"))
)

# Enlever les espaces en trop dans les colonnes texte
colonnes_texte = [c for c, t in produits.dtypes if t == "string" and c not in ("motif_rejet", "_corrupt_record")]
for c in colonnes_texte:
    produits = produits.withColumn(c, F.trim(F.col(c)))

# Vides déguisés -> NULL
for c in ["i_size", "i_units"]:
    produits = produits.withColumn(c, F.when(F.col(c).isin("N/A", "Unknown", ""), None).otherwise(F.col(c)))

produits_propres = produits.filter(F.col("motif_rejet").isNull()) \
                           .drop("motif_rejet", "nb_meme_id", "_corrupt_record", "i_container")
produits_rejetes = produits.filter(F.col("motif_rejet").isNotNull())


In [4]:
# Règles sur les ventes
ids_produits = produits_propres.select(F.col("i_item_sk").alias("ss_item_sk"), F.lit(True).alias("produit_connu"))

ventes = (
    ventes_brut
    .dropDuplicates()
    .join(ids_produits, on="ss_item_sk", how="left")
    .withColumn("motif_rejet",
        F.when(F.col("_corrupt_record").isNotNull(), "type incorrect")
         .when(F.col("ss_item_sk").isNull() | F.col("ss_ticket_number").isNull()
               | F.col("ss_sold_date_sk").isNull() | F.col("ss_store_sk").isNull(), "identifiant manquant")
         .when(F.col("produit_connu").isNull(), "produit inconnu")
         .when(F.col("ss_quantity") <= 0, "quantité <= 0")
         .when((F.col("ss_sales_price") < 0) | (F.col("ss_net_paid") < 0), "montant négatif"))
    .withColumn("prix_zero", F.col("ss_sales_price") == 0)
)

colonnes_ventes = [c for c in schema_ventes.fieldNames() if c != "_corrupt_record"] + ["prix_zero"]
ventes_propres = ventes.filter(F.col("motif_rejet").isNull()).select(colonnes_ventes)
ventes_rejetees = ventes.filter(F.col("motif_rejet").isNotNull())


In [5]:
# Bilan du nettoyage
print("produits : avant", produits_brut.count(), "| gardés", produits_propres.count(),
      "| mis à l'écart", produits_rejetes.count())
produits_rejetes.groupBy("motif_rejet").count().show()

print("ventes : avant", ventes_brut.count(), "| après doublons exacts", ventes_brut.dropDuplicates().count(),
      "| gardées", ventes_propres.count(), "| mises à l'écart", ventes_rejetees.count())
ventes_rejetees.groupBy("motif_rejet").count().show()

print("ventes à prix 0 gardées et repérées :", ventes_propres.filter("prix_zero").count())
print("i_size NULL après la règle :", produits_propres.filter(F.col("i_size").isNull()).count(),
      "| i_units NULL :", produits_propres.filter(F.col("i_units").isNull()).count())

ventes_propres.printSchema()
produits_propres.printSchema()

produits : avant 17820 | gardés 17820 | mis à l'écart 0


+-----------+-----+
|motif_rejet|count|
+-----------+-----+
+-----------+-----+



ventes : avant 667579 | après doublons exacts 667579 | gardées 667579 | mises à l'écart 0


+-----------+-----+
|motif_rejet|count|
+-----------+-----+
+-----------+-----+



ventes à prix 0 gardées et repérées : 3324


i_size NULL après la règle : 2502 | i_units NULL : 1767
root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_

## Étape 5 – Transformations : jointure ventes + produits et agrégation

In [6]:
# Vérifier la clé avant la jointure
print("produits - lignes :", produits_propres.count(),
      "| i_item_sk distincts :", produits_propres.select("i_item_sk").distinct().count())

ventes_sans_produit = ventes_propres.join(
    produits_propres, ventes_propres.ss_item_sk == produits_propres.i_item_sk, "left_anti")
print("ventes dont le produit n'existe pas dans item :", ventes_sans_produit.count())
print("produits différents vendus :", ventes_propres.select("ss_item_sk").distinct().count())

produits - lignes : 17820 | i_item_sk distincts : 17820


ventes dont le produit n'existe pas dans item : 0


produits différents vendus : 17820


In [7]:
# Jointure ventes + produits
colonnes_produit = produits_propres.select(
    "i_item_sk", "i_item_id", "i_product_name", "i_category", "i_class", "i_brand")

ventes_enrichies = ventes_propres.join(
    colonnes_produit, ventes_propres.ss_item_sk == colonnes_produit.i_item_sk, "left")

print("Lignes avant :", ventes_propres.count())
print("Lignes après :", ventes_enrichies.count())
print("Ventes non rapprochées :", ventes_enrichies.filter(F.col("i_item_sk").isNull()).count())
ventes_enrichies.select("ss_ticket_number", "ss_item_sk", "i_category", "i_class",
                        "ss_quantity", "ss_net_paid").show(5)

Lignes avant : 667579


Lignes après : 667579


Ventes non rapprochées : 0


+----------------+----------+----------+--------------------+-----------+-----------+
|ss_ticket_number|ss_item_sk|i_category|             i_class|ss_quantity|ss_net_paid|
+----------------+----------+----------+--------------------+-----------+-----------+
|              35|      3560|     Music|  Dance & Electronic|         31|    2296.48|
|              41|     11504|     Books|Education & Refer...|         81|    4221.72|
|              46|      4677|     Music|                Rock|         90|    7504.20|
|              50|      1072|     Music|    Alternative Rock|         35|     714.70|
|              64|      9835|     Books|Education & Refer...|         10|     497.70|
+----------------+----------+----------+--------------------+-----------+-----------+
only showing top 5 rows


In [8]:
# Agrégation : chiffre d'affaires par catégorie
ca_total = ventes_propres.agg(F.sum("ss_net_paid")).first()[0]

ca_par_categorie = (
    ventes_enrichies
    .groupBy("i_category")
    .agg(
        F.sum("ss_net_paid").alias("chiffre_affaires"),
        F.sum("ss_quantity").alias("quantite_vendue"),
        F.count("*").alias("nombre_lignes_vente"),
        F.countDistinct("ss_item_sk").alias("nombre_produits_vendus")
    )
    .withColumnRenamed("i_category", "categorie")
    .withColumn("part_ca_pct", F.round(F.col("chiffre_affaires") / F.lit(ca_total) * 100, 2))
    .orderBy(F.desc("chiffre_affaires"))
)
ca_par_categorie.show(truncate=False)

+------------------------+----------------+---------------+-------------------+----------------------+-----------+
|categorie               |chiffre_affaires|quantite_vendue|nombre_lignes_vente|nombre_produits_vendus|part_ca_pct|
+------------------------+----------------+---------------+-------------------+----------------------+-----------+
|Music                   |519666693.84    |15238374       |301236             |8143                  |45.17      |
|Books                   |307151503.25    |9007511        |177671             |5335                  |26.70      |
|Movies & TV             |74735891.89     |2189279        |43376              |1286                  |6.50       |
|Clothing & Accessories  |56754855.82     |1640550        |32372              |610                   |4.93       |
|Electronics             |55715087.49     |1632098        |32619              |692                   |4.84       |
|Home & Kitchen          |41260511.11     |1226839        |24380              |4

In [9]:
# Contrôler que l'agrégation ne perd rien
print("CA total des ventes  :", ca_total)
print("Somme des catégories :", ca_par_categorie.agg(F.sum("chiffre_affaires")).first()[0])
print("Lignes des ventes    :", ventes_propres.count())
print("Somme des lignes     :", ca_par_categorie.agg(F.sum("nombre_lignes_vente")).first()[0])
ca_par_categorie.printSchema()

CA total des ventes  : 1150529586.37


Somme des catégories : 1150529586.37


Lignes des ventes    : 667579


Somme des lignes     : 667579
root
 |-- categorie: string (nullable = true)
 |-- chiffre_affaires: decimal(17,2) (nullable = true)
 |-- quantite_vendue: long (nullable = true)
 |-- nombre_lignes_vente: long (nullable = false)
 |-- nombre_produits_vendus: long (nullable = false)
 |-- part_ca_pct: decimal(24,2) (nullable = true)

